In [ ]:
# Import Libraries

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Load Data 

folders = sorted([
    folder for folder in os.listdir(".")
    if folder.startswith("call-report-data-")
])

folders

In [ ]:
#Process Data 

def process_quarter(folder):
    # Load files
    fs220 = pd.read_csv(
        os.path.join(folder, "FS220.txt"),
        low_memory=False
    )

    fs220a = pd.read_csv(
        os.path.join(folder, "FS220A.txt"),
        low_memory=False
    )

    foicu = pd.read_csv(
        os.path.join(folder, "FOICU.txt"),
        low_memory=False
    )

    # Select financial columns
    fs220_selected = fs220[
        [
            "CU_NUMBER",
            "CYCLE_DATE",
            "ACCT_010",
            "ACCT_018",
            "ACCT_025A",
            "ACCT_025B",
            "ACCT_083"
        ]
    ].copy()

    fs220a_selected = fs220a[
        [
            "CU_NUMBER",
            "CYCLE_DATE",
            "Acct_661A",
            "Acct_997",
            "Acct_998"
        ]
    ].copy()

    # Merge financial files
    quarter = pd.merge(
        fs220_selected,
        fs220a_selected,
        on=["CU_NUMBER", "CYCLE_DATE"],
        how="left"
    )

    # Rename financial columns
    quarter = quarter.rename(columns={
        "CU_NUMBER": "Credit_Union_Number",
        "CYCLE_DATE": "Quarter",
        "ACCT_010": "Total_Assets",
        "ACCT_018": "Shares_Deposits",
        "ACCT_025A": "Number_of_Loans",
        "ACCT_025B": "Total_Loans",
        "ACCT_083": "Members",
        "Acct_661A": "Net_Income",
        "Acct_997": "Total_Net_Worth",
        "Acct_998": "Net_Worth_Ratio"
    })

    # Select credit union information
    foicu_selected = foicu[
        [
            "CU_NUMBER",
            "CYCLE_DATE",
            "CU_NAME",
            "CITY",
            "STATE",
            "CU_TYPE",
            "YEAR_OPENED",
            "Peer_Group"
        ]
    ].copy()

    # Merge credit union information
    quarter = pd.merge(
        quarter,
        foicu_selected,
        left_on=["Credit_Union_Number", "Quarter"],
        right_on=["CU_NUMBER", "CYCLE_DATE"],
        how="left"
    )

    quarter = quarter.drop(
        columns=["CU_NUMBER", "CYCLE_DATE"]
    )

    quarter = quarter.rename(columns={
        "CU_NAME": "Credit_Union_Name",
        "CITY": "City",
        "STATE": "State",
        "CU_TYPE": "Credit_Union_Type",
        "YEAR_OPENED": "Year_Opened"
    })

    # Convert date
    quarter["Quarter"] = pd.to_datetime(quarter["Quarter"])

    # Convert net worth ratio to percentage
    quarter["Net_Worth_Ratio"] = quarter["Net_Worth_Ratio"] / 100

    return quarter

In [ ]:
#Combine Data

all_quarters = []

for folder in folders:
    quarter_data = process_quarter(folder)
    all_quarters.append(quarter_data)

combined_df = pd.concat(
    all_quarters,
    ignore_index=True
)

combined_df.shape

In [ ]:
# Check all dates are loaded

print(
    "First qtr:",
    combined_df["Quarter"].min()
)

print(
    "Last qtr:",
    combined_df["Quarter"].max()
)

In [ ]:
combined_df["Quarter"].value_counts().sort_index()

In [ ]:
# Check data quality 

print("Shape:", combined_df.shape)

print("\nMissing values:")
print(combined_df.isnull().sum())

print("\nDuplicate rows:")
print(combined_df.duplicated().sum())

print("\nDuplicate credit union-quarter records:")
print(
    combined_df.duplicated(
        subset=["Credit_Union_Number", "Quarter"]
    ).sum()
)

In [ ]:
# Check data types

combined_df.info()

In [ ]:
# Summary statistics

combined_df[
    [
        "Total_Assets",
        "Shares_Deposits",
        "Number_of_Loans",
        "Total_Loans",
        "Members",
        "Net_Income",
        "Total_Net_Worth",
        "Net_Worth_Ratio"
    ]
].describe()

In [ ]:
# Look @ small credit unions based on assets, loans, members, networth ratio

combined_df.nsmallest(
    10, "Total_Assets"
)[
    [
        "Credit_Union_Number",
        "Credit_Union_Name",
        "Quarter",
        "State",
        "Total_Assets",
        "Total_Loans",
        "Members",
        "Net_Worth_Ratio"
    ]
]

In [ ]:
#Check zeros

zero_counts = {
    "Total_Assets": (combined_df["Total_Assets"] == 0).sum(),
    "Shares_Deposits": (combined_df["Shares_Deposits"] == 0).sum(),
    "Number_of_Loans": (combined_df["Number_of_Loans"] == 0).sum(),
    "Total_Loans": (combined_df["Total_Loans"] == 0).sum(),
    "Members": (combined_df["Members"] == 0).sum()
}

zero_counts

In [ ]:
# Compare membership thresholds

thresholds = [10, 50, 100, 500, 1000]

for threshold in thresholds:
    count = (combined_df["Members"] < threshold).sum()
    percent = count / len(combined_df) * 100
    
    print(
        f"Below {threshold:,} members: "
        f"{count} ({percent:.2f}%)"
    )

In [ ]:
# Membership distribution

combined_df["Members"].describe(
    percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]
)

In [ ]:
# previous quarter values

combined_df = combined_df.sort_values(
    ["Credit_Union_Number", "Quarter"]
).reset_index(drop=True)

combined_df["Prev_Qr"] = (
    combined_df.groupby("Credit_Union_Number")["Quarter"]
    .shift(1)
)

combined_df["Prev_Tot_Loans"] = (
    combined_df.groupby("Credit_Union_Number")["Total_Loans"]
    .shift(1)
)

In [ ]:
# Show consecutive quarters

combined_df["Quarter_Number"] = (
    combined_df["Quarter"].dt.year * 4
    + combined_df["Quarter"].dt.quarter
)

combined_df["Previous_Quarter_Number"] = (
    combined_df
    .groupby("Credit_Union_Number")["Quarter_Number"]
    .shift(1)
)

combined_df["Consecutive_Quarter"] = (
    combined_df["Quarter_Number"]
    - combined_df["Previous_Quarter_Number"]
    == 1
)

In [ ]:
# Calcualte loan growth 

combined_df["Loan_Growth"] = np.where(
    (combined_df["Consecutive_Quarter"]) &
    (combined_df["Prev_Tot_Loans"] > 0),

    (
        (combined_df["Total_Loans"]
         - combined_df["Prev_Tot_Loans"])
        / combined_df["Prev_Tot_Loans"]
    ) * 100,

    np.nan
)

In [ ]:
# only keep 100+ members

analysis_df = combined_df[
    combined_df["Members"] >= 100
].copy()

In [ ]:
removed = len(combined_df) - len(analysis_df)

print("Original #:", len(combined_df))
print("Analysis #:", len(analysis_df))
print("# Removed:", removed)

In [ ]:
# Loan growth distribution 

analysis_df["Loan_Growth"].describe(
    percentiles=[.01, .05, .25, .50, .75, .95, .99]
)

In [ ]:
#add outlier bounds 

lower_bound = analysis_df["Loan_Growth"].quantile(0.01)
upper_bound = analysis_df["Loan_Growth"].quantile(0.99)

print("1st percentile:", lower_bound)
print("99th percentile:", upper_bound)

In [ ]:
# Remove extreme loan growth values

growth_df = analysis_df[
    analysis_df["Loan_Growth"].between(
        lower_bound,
        upper_bound
    )
].copy()

In [ ]:
print("Before trim:", analysis_df["Loan_Growth"].notna().sum())
print("After trim:", len(growth_df))

In [ ]:
growth_df["Loan_Growth"].describe()

In [ ]:
# Review loan growth distribution after trimming 

growth_df["Loan_Growth"].describe(
    percentiles=[.01, .05, .25, .50, .75, .95, .99]
)

In [ ]:
#EDA

# Average and median loan growth by quarter

quarterly_growth = (
    growth_df
    .groupby("Quarter")["Loan_Growth"]
    .agg(["mean", "median"])
    .reset_index()
)

quarterly_growth

In [ ]:
quarterly_growth["Quarter_Label"] = (
    quarterly_growth["Quarter"].dt.year.astype(str)
    + " Q"
    + quarterly_growth["Quarter"].dt.quarter.astype(str)
)

plt.figure(figsize=(10, 5))

plt.plot(
    quarterly_growth["Quarter_Label"],
    quarterly_growth["median"],
    marker="o"
)

plt.axhline(0, linestyle="--")

plt.title("Median Loan Growth by Quarter")
plt.xlabel("Quarter")
plt.ylabel("Loan Growth (%)")

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()